# Chart Recommender Framework — Layer 1 Test Notebook

Purpose: Run the 19 prepared test cases through the REAL chart-recommendation
function from your cloned repo, and log results into MLflow.

Plain Python — no pytest, no Playwright required. Only needs:
`pip install mlflow pandas openpyxl`

**Before running:** update the two TODO cells below —
1. `call_chart_recommender()` — point this at the real function in your cloned repo.
2. `log_results_to_mlflow()` — ideally swap in your project's existing confusion-matrix logger.


In [ ]:
# --- Imports ---
import pandas as pd
import mlflow

In [ ]:
# --- Config ---
TEST_DATASET_PATH = "Chart_Recommender_Test_Dataset.csv"   # or the .xlsx file
MLFLOW_EXPERIMENT_NAME = "chart_recommender_evaluation"

## Step 1 — Hook up the real chart recommender function (TODO)

In [ ]:
def call_chart_recommender(user_question: str, country: str = None) -> str:
    """
    TODO: Replace this with a call to your real chart recommendation function
    from the cloned repo. Example (adjust the import path and arguments to
    match your actual project structure):

        from src.chart_recommender.core import recommend_chart_type
        return recommend_chart_type(user_question, country=country)

    For now this raises an error so you don\'t accidentally run the notebook
    without wiring it up first.
    """
    raise NotImplementedError(
        "Wire this up to your real chart recommendation function before running."
    )

## Step 2 — Log results into MLflow (fallback logger provided; swap in your project's existing confusion-matrix logger if you find one)

In [ ]:
def log_results_to_mlflow(expected_list, actual_list, question_list):
    """
    Preferred: use your project\'s existing confusion-matrix MLflow logger.
    Example:

        from src.mlflow_utils.confusion_matrix_logger import log_confusion_matrix
        log_confusion_matrix(expected_list, actual_list)
        return

    Fallback (used below): logs each test case as its own MLflow run with a
    simple pass/fail metric, plus an overall accuracy metric.
    """
    correct = 0
    for question, expected, actual in zip(question_list, expected_list, actual_list):
        with mlflow.start_run(run_name=question[:40]):
            mlflow.log_param("user_question", question)
            mlflow.log_param("expected_chart_type", expected)
            mlflow.log_param("actual_chart_type", actual)
            is_match = (str(expected).strip().lower() == str(actual).strip().lower())
            mlflow.log_metric("match", int(is_match))
            if is_match:
                correct += 1

    accuracy = correct / len(expected_list) if expected_list else 0
    with mlflow.start_run(run_name="OVERALL_SUMMARY"):
        mlflow.log_metric("overall_accuracy", accuracy)
        mlflow.log_metric("total_cases", len(expected_list))
        mlflow.log_metric("correct_cases", correct)

    print(f"Overall accuracy: {accuracy:.2%} ({correct}/{len(expected_list)})")

## Load the test dataset

In [ ]:
df = pd.read_csv(TEST_DATASET_PATH)   # use pd.read_excel(..., sheet_name=..., header=3) if using the .xlsx version
df = df.dropna(subset=["Family Chart"])
df.head()

## Run the test loop

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

expected_list = []
actual_list = []
question_list = []
results_summary = []

for _, row in df.iterrows():
    question = row["User Question"]
    expected_chart_type = row["Chart Type"]
    country = row["Country Selection"]

    try:
        actual_chart_type = call_chart_recommender(question, country=country)
    except NotImplementedError:
        raise  # stop immediately -- you haven\'t wired up the function yet
    except Exception as e:
        actual_chart_type = f"ERROR: {e}"

    expected_list.append(expected_chart_type)
    actual_list.append(actual_chart_type)
    question_list.append(question)

    match = str(expected_chart_type).strip().lower() == str(actual_chart_type).strip().lower()
    results_summary.append({
        "question": question,
        "expected": expected_chart_type,
        "actual": actual_chart_type,
        "match": match,
    })

log_results_to_mlflow(expected_list, actual_list, question_list)

## Review results

In [ ]:
summary_df = pd.DataFrame(results_summary)
summary_df.to_csv("chart_recommender_test_summary.csv", index=False)
summary_df

## Next step

Open the MLflow Tracking UI (usually `mlflow ui` in a terminal, or however your
project already launches it) and confirm a new run appears for each test case
under the `chart_recommender_evaluation` experiment, plus one `OVERALL_SUMMARY`
run with the total accuracy.